In [ ]:
# Bring your own data: this notebook grades drb_quick_scan_results.jsonl in the working directory, as written
# by drb-quick-scan.ipynb (JSONL, one {"id", "prompt", "article"} object per line, ids and prompts from the
# Deep Research Bench query file). No data ships with this repository.
import os, re, subprocess, shutil, pathlib

# --- Grader revision ----------------------------------------------------------
# GRADER_REVISION is the grader checkout the results are graded with: the full 40-character commit SHA of
# Ayanami0730/deep_research_bench. This is the current upstream commit at the time of the pin (2026-11-05).
# Every grading step refuses to run while it is None or is not
# a full lowercase SHA, or while the checkout is not exactly at it. The notebook never moves the checkout itself.
GRADER_REVISION = "f2735b5c3636759c22f9e936c0232de8bf545b67"
GRADER_REPO_URL = "https://github.com/Ayanami0730/deep_research_bench.git"
GRADER_DIR = "deep_research_bench"

# --- Cost bound ----------------------------------------------------------------
# Spend policy: grading stays off until every paid call the grader makes can be metered or bounded before
# it is dispatched. GRADER_COST_BOUNDED may be set to True only after that is true for every internal paid grader call
# (RACE scoring, cleaning, retries, FACT steps). Until then every grading entry in this notebook refuses.
GRADER_COST_BOUNDED = False


def require_bounded_grading():
    if GRADER_COST_BOUNDED is not True:
        raise RuntimeError(
            "Refusing to grade: the grader makes its own model calls in a subprocess (RACE scoring, cleaning, retries "
            "and the FACT steps), and this notebook cannot meter or cap any of them before they are sent. The flat "
            "per-step reservation is not an upper bound: a single call with a 64000-token output lists at about 7.20 USD "
            "on openrouter before any input cost, and scoring and cleaning retry. Grading stays off until those calls "
            "are bounded. Nothing was installed, copied or sent.")


def require_grader_revision():
    if GRADER_REVISION is None or not re.fullmatch(r"[0-9a-f]{40}", str(GRADER_REVISION)):
        raise RuntimeError("GRADER_REVISION is not a full 40-character commit SHA. Refusing to grade. "
                           "The operator must set it first.")


def verify_grader_tree_clean():
    """Read-only working-tree integrity check for the grader checkout. The checkout verification calls it,
    so every grader entry point and every grader subprocess is covered. It refuses when a tracked file is modified,
    deleted, staged or has a changed mode, when any untracked file (ignored files included) is not a plain data file,
    or when the checkout holds a nested .git or a symlink. Allowed untracked files: .jsonl or .json under
    data/test_data/raw_data/ and data/test_data/cleaned_data/, and .jsonl, .json, .txt or .log under results/, none with an executable bit. Nothing is
    cleaned, fetched or checked out here, and git is only asked to read."""
    import stat
    fix = (f"What to do by hand: inspect each item listed above, then run  git -C {GRADER_DIR} status  and look at the "
           f"tree. If you did not make the change yourself, re-clone {GRADER_DIR}, or run  git -C {GRADER_DIR} checkout "
           f"--detach {GRADER_REVISION}  yourself (that drops tracked edits), and remove stray untracked files yourself. "
           "This notebook never cleans or checks out anything itself.")
    keep = ("PATH", "HOME", "USERPROFILE", "XDG_CONFIG_HOME", "SYSTEMROOT", "TMPDIR")
    env = {k: os.environ[k] for k in keep if k in os.environ}
    env.update(LC_ALL="C", GIT_TERMINAL_PROMPT="0")
    base = ["git", "-C", GRADER_DIR, "--no-optional-locks", "-c", "core.fsmonitor=false", "-c", "core.fileMode=true"]

    def git(*args):
        try:
            out = subprocess.run(base + list(args), capture_output=True, check=True, env=env, timeout=120).stdout
        except (OSError, subprocess.SubprocessError) as e:
            raise RuntimeError(f"Refusing to grade: cannot read the working tree state of {GRADER_DIR} "
                               f"({type(e).__name__}). {fix}")
        return out.encode() if isinstance(out, str) else out

    def entries(raw):
        return [x.decode("utf-8", "surrogateescape") for x in raw.split(b"\0") if x]

    problems = []
    for e in entries(git("status", "--porcelain=v1", "-z", "--untracked-files=no", "--ignore-submodules=none")):
        problems.append(f"tracked file changed (modified, deleted, staged or mode changed): {e}")
    for e in entries(git("ls-files", "-v", "-z")):
        if not e.startswith("H "):
            problems.append(f"tracked file hidden from status (assume-unchanged, skip-worktree or unmerged): {e[2:]}")
    for p in entries(git("ls-files", "-z", "--others")):
        parts = p.split("/")
        ext = os.path.splitext(parts[-1])[1]
        if "" in parts or ".." in parts or ".git" in parts:
            problems.append(f"nested repository or unsafe path: {p}")
            continue
        ok = ((p.startswith("data/test_data/raw_data/") and ext in (".jsonl", ".json")) or
              (p.startswith("data/test_data/cleaned_data/") and ext in (".jsonl", ".json")) or
              (p.startswith("results/") and ext in (".jsonl", ".json", ".txt", ".log")))
        if not ok:
            problems.append(f"untracked file that is not an allowed data file: {p}")
            continue
        try:
            st = os.lstat(os.path.join(GRADER_DIR, *parts))
        except OSError:
            problems.append(f"untracked file that cannot be inspected: {p}")
            continue
        if stat.S_ISLNK(st.st_mode):
            problems.append(f"untracked symlink: {p}")
        elif not stat.S_ISREG(st.st_mode):
            problems.append(f"untracked entry that is not a regular file: {p}")
        elif st.st_mode & 0o111:
            problems.append(f"untracked data file with an executable bit: {p}")
    for root, dirs, files in os.walk(GRADER_DIR):
        if root == GRADER_DIR and ".git" in dirs:
            dirs.remove(".git")
        for n in list(dirs) + list(files):
            if n == ".git":
                problems.append(f"nested .git: {os.path.join(root, n)}")
        if ".git" in dirs:
            dirs.remove(".git")
    if problems:
        shown = "\n".join("  - " + x for x in problems[:20]) + (f"\n  ... and {len(problems) - 20} more" if len(problems) > 20 else "")
        raise RuntimeError(f"Refusing to grade: the grader checkout in {GRADER_DIR} differs from the pinned commit in a "
                           f"way that could run different code with this process's environment. Found:\n{shown}\n{fix}")


def ensure_grader_checkout():
    """Read-only check that the grader checkout is at GRADER_REVISION, that its working tree is clean (see
    verify_grader_tree_clean), and that its origin is GRADER_REPO_URL.
    Nothing is cloned, fetched or checked out here. On any mismatch it refuses and prints the commands to run by hand."""
    require_grader_revision()
    fix = (f"  git clone {GRADER_REPO_URL} {GRADER_DIR}      (only if {GRADER_DIR} does not exist)\n"
           f"  git -C {GRADER_DIR} remote set-url origin {GRADER_REPO_URL}\n"
           f"  git -C {GRADER_DIR} fetch origin {GRADER_REVISION}\n"
           f"  git -C {GRADER_DIR} checkout --detach {GRADER_REVISION}")
    try:
        head = subprocess.run(["git", "-C", GRADER_DIR, "rev-parse", "HEAD"], capture_output=True, text=True,
                              check=True).stdout.strip()
        origin = subprocess.run(["git", "-C", GRADER_DIR, "remote", "get-url", "origin"], capture_output=True,
                                text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError) as e:
        raise RuntimeError(f"Refusing to grade: cannot read the grader checkout in {GRADER_DIR} ({e}). "
                           f"The operator must run by hand:\n{fix}")
    if head != GRADER_REVISION or origin != GRADER_REPO_URL:
        raise RuntimeError(f"Refusing to grade: the grader checkout is at {head} with origin {origin}, but it must be "
                           f"at {GRADER_REVISION} with origin {GRADER_REPO_URL}. The operator must run by hand:\n{fix}")
    verify_grader_tree_clean()


require_bounded_grading()
ensure_grader_checkout()

# --- Spend guard -----------------------------------------------------------
# Each grader step starts paid model calls made by the grader process. Spend policy: every grader attempt (retries and
# failed attempts included) is reserved at JOB_WORST_CASE_USD (default 2.00, overridable by an env var of the same name)
# before it is started and settled to it after. RUN_BUDGET_USD (default 25.00, env var of the same name) caps the run: a
# step that would take reserved plus settled spend past it is refused, and the run stops. Both must be finite numbers
# above 0 or the notebook refuses to start. The ledger root is SPEND_LEDGER_DIR (default <repo root>/spend_ledger) and
# the one runner.lock in it is shared by every notebook, so only one paid run exists at a time. The grader builds its
# own requests in its own process, so provider.max_price and usage cannot be added here, and this notebook has no
# max_tokens to multiply by a price; each step is reserved as a job. Status polls are not paid.
import atexit, json, math, os, threading, uuid
from datetime import datetime, timezone


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD and JOB_WORST_CASE_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)
RUN_ID = None                # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
JOB_WORST_CASE_USD = _env_usd("JOB_WORST_CASE_USD", 2.00)   # reserved for every grader attempt, in USD


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _spend_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_ROOT = _spend_root()


class BudgetExceeded(RuntimeError):
    pass


# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One runner at a time across all notebooks: one exclusive lock file in the shared ledger root. A second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock is never cleared automatically.
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner (any notebook) may be using the shared ledger, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)

_ledger_lock = threading.Lock()
_ledger = {}                 # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False       # set by a refused submit; no later submit is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)


def ledger_total():          # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def reserve_job(purpose):
    """Reserve one submit at JOB_WORST_CASE_USD before it is sent. Raises BudgetExceeded if no worst case is set,
    if the run is halted, or if the reservation would take the run past RUN_BUDGET_USD."""
    global _budget_halted
    if JOB_WORST_CASE_USD is None or not math.isfinite(JOB_WORST_CASE_USD) or JOB_WORST_CASE_USD <= 0:
        raise BudgetExceeded("not sent: JOB_WORST_CASE_USD is not a positive amount. The operator must set it first.")
    wc = round(float(JOB_WORST_CASE_USD), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so the {purpose} submit is refused")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "purpose": purpose, "worst_case": wc,
                 "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {purpose} submit, worst case ${wc:.4f}, would take the run past "
                         f"RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def settle_job(aid):
    """Settle one submit at its worst case. A submit is charged whether or not the job is accepted."""
    with _ledger_lock:
        e = _ledger[aid]
        e.update(state="settled", cost=e["worst_case"], cost_source="worst_case")
        _ledger_write({"id": aid, "state": "settled", "cost": e["worst_case"], "cost_source": "worst_case"})


print(f"Spend ledger: run {RUN_ID}, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}, charged so far ${ledger_total():.4f}")

# Confirmed listed prices in USD per million tokens, (input, output), the highest that can apply, retrieved 2026-11-05.
# Keys are route|model. Sources: https://openrouter.ai/api/v1/models/openai/gpt-5.5/endpoints,
# https://openrouter.ai/api/v1/models/openai/gpt-5.6-luna/endpoints and https://openrouter.ai/openai/gpt-5.4-mini
# (openrouter_http); https://developers.openai.com/api/docs/models/gpt-5.5 and
# https://developers.openai.com/api/docs/pricing (openai_direct).
# gpt-5.4-mini on openai_direct has no confirmed price, so it is left out on purpose: the FACT steps refuse on
# LLM_BACKEND=openai unless FACT_MODEL is set to a model that is listed here. Jina scrape calls are not model calls.
PRICE = {
    "openrouter_http|openai/gpt-5.5": (25.0, 112.5),
    "openrouter_http|openai/gpt-5.6-luna": (0.8, 3.6),
    "openrouter_http|openai/gpt-5.4-mini": (0.825, 4.95),
    "openai_direct|gpt-5.5": (10.0, 45.0),
    "openai_direct|gpt-5.6-luna": (0.4, 1.8),
}
# Output cap per grader call, sent to the grader as MAX_OUTPUT_TOKENS (the grader's own default is also 64000).
GRADER_MAX_OUTPUT_TOKENS = 64000
# At the pinned commit the grader reads LLM_BACKEND and calls up to three models: RACE_MODEL (scoring) and CLEAN_MODEL
# (cleaning) inside deepresearch_bench_race.py, and FACT_MODEL in the utils.extract, utils.deduplicate and
# utils.validate steps. Each backend reads exactly one key variable, by this name. Per backend:
# (route, key variable name, default base URL, default RACE_MODEL, default CLEAN_MODEL, default FACT_MODEL).
_GRADER_BACKENDS = {
    "openrouter": ("openrouter_http", "OPENROUTER_API_KEY", "https://openrouter.ai/api/v1",
                   "openai/gpt-5.5", "openai/gpt-5.6-luna", "openai/gpt-5.4-mini"),
    "openai": ("openai_direct", "OPENAI_API_KEY", "https://api.openai.com/v1",
               "gpt-5.5", "gpt-5.6-luna", "gpt-5.4-mini"),
}
_BASE_URL_VARS = {"openrouter": "OPENROUTER_BASE_URL", "openai": "OPENAI_BASE_URL"}


def require_grader_prices(stages=("race", "clean")):
    """Refuse before any paid call unless the backend is known, its key variable is present (checked by name only,
    the value is never printed), the base URL is the official one, and every model the grader will call in the given
    stages has a confirmed price in PRICE. Returns the environment to pass to the grader so it calls exactly the
    models that were checked."""
    backend = os.environ.get("LLM_BACKEND", "openrouter").lower()
    if backend not in _GRADER_BACKENDS:
        raise RuntimeError(f"Refusing to grade: LLM_BACKEND={backend!r} is not one of {sorted(_GRADER_BACKENDS)}, "
                           "so no price can be confirmed.")
    route, key_var, base_url, d_race, d_clean, d_fact = _GRADER_BACKENDS[backend]
    if not os.environ.get(key_var, "").strip():
        raise RuntimeError(f"Refusing to grade: {key_var} is not set or is blank. The grader on LLM_BACKEND={backend} "
                           f"reads its key from exactly that variable name. Set {key_var} before running.")
    base_var = _BASE_URL_VARS[backend]
    if base_var in os.environ and os.environ[base_var] != base_url:
        raise RuntimeError(f"Refusing to grade: {base_var} is set to something other than {base_url}. The prices in "
                           f"PRICE are for the official host, and the key would be sent to the other host. Unset {base_var}.")
    wanted = {"race": ("RACE_MODEL", d_race), "clean": ("CLEAN_MODEL", d_clean), "fact": ("FACT_MODEL", d_fact)}
    models = {}
    for stage in stages:
        var, default = wanted[stage]
        models[var] = os.environ.get(var, default)
    for var, m in models.items():
        if f"{route}|{m}" not in PRICE:
            raise RuntimeError(f"Refusing to grade: no confirmed price for {var}={m!r} on route {route}. Add "
                               f"'{route}|{m}' to PRICE in this notebook with its source URL and date, or set {var} "
                               "to a model that is listed in PRICE.")
    return dict(LLM_BACKEND=backend, MAX_OUTPUT_TOKENS=str(GRADER_MAX_OUTPUT_TOKENS), **models)


# The grader runs in a subprocess, so its model calls cannot be metered one by one here. Each grader step is therefore
# a flat JOB_WORST_CASE_USD reservation (default 2.00, spend policy). It is not an upper bound: one gpt-5.5 call at
# 64000 output tokens on openrouter lists at about 7.2 USD, and scoring and cleaning retry. RUN_BUDGET_USD caps the run.
# The spend policy is to refuse until bounded (GRADER_COST_BOUNDED), so no step runs today.
_grader_env = dict(os.environ, **require_grader_prices(), PYTHONDONTWRITEBYTECODE="1")
subprocess.run(["pip", "install", "-r", "deep_research_bench/requirements.txt", "--quiet"], check=True)

shutil.copy("drb_quick_scan_results.jsonl", "deep_research_bench/data/test_data/raw_data/quick-scan-v3.jsonl")

require_bounded_grading()
ensure_grader_checkout()
_grader_env = dict(os.environ, **require_grader_prices(), PYTHONDONTWRITEBYTECODE="1")
with open("grading_log.txt", "w") as log:
    aid = reserve_job("grade")
    try:
        r = subprocess.run(["python", "deepresearch_bench_race.py", "quick-scan-v3", "--output_dir", "results/race/quick-scan-v3"], cwd="deep_research_bench", stdout=log, stderr=subprocess.STDOUT, env=_grader_env, timeout=7200)
    finally:
        settle_job(aid)

out = ""
out += f"Exit code: {r.returncode}\n"
rpath = "deep_research_bench/results/race/quick-scan-v3/race_result.txt"
if pathlib.Path(rpath).exists():
    out += open(rpath).read()
else:
    out += "No results file. Download grading_log.txt to debug."
pathlib.Path("grading_status.txt").write_text(out)
out

In [ ]:
require_bounded_grading()
import os, subprocess, pathlib

cwd = "deep_research_bench"
model = "quick-scan-v3"
raw = f"data/test_data/raw_data/{model}.jsonl"
out_dir = f"results/fact/{model}"
query = "data/prompt_data/query.jsonl"
n = "10"   # --n_total_process: grader-internal processes inside each of steps 1 to 4. This notebook has no worker pool;
           # each step is reserved once at JOB_WORST_CASE_USD, so set that to cover a whole step.

ensure_grader_checkout()
# FACT steps call only FACT_MODEL. On LLM_BACKEND=openai the default gpt-5.4-mini has no confirmed price, so this
# refuses there unless FACT_MODEL is set to a model listed in PRICE (that changes the fact checker). openrouter works.
_grader_env = dict(os.environ, **require_grader_prices(("fact",)), PYTHONDONTWRITEBYTECODE="1")
# Only utils.scrape needs JINA_API_KEY. Checked by name here so nothing is reserved before a missing key is known.
if not os.environ.get("JINA_API_KEY", "").strip():
    raise RuntimeError("Refusing to grade: JINA_API_KEY is not set or is blank. The scrape step reads its key from "
                       "exactly that variable name. Set JINA_API_KEY before running.")
if not pathlib.Path(f"{cwd}/{raw}").exists():
    raise RuntimeError(f"Refusing to grade: {cwd}/{raw} does not exist. Run the previous cell first.")
pathlib.Path(f"{cwd}/{out_dir}").mkdir(parents=True, exist_ok=True)

steps = [
    ["python", "-u", "-m", "utils.extract", "--raw_data_path", raw, "--output_path", f"{out_dir}/extracted.jsonl", "--query_data_path", query, "--n_total_process", n],
    ["python", "-u", "-m", "utils.deduplicate", "--raw_data_path", f"{out_dir}/extracted.jsonl", "--output_path", f"{out_dir}/deduplicated.jsonl", "--query_data_path", query, "--n_total_process", n],
    ["python", "-u", "-m", "utils.scrape", "--raw_data_path", f"{out_dir}/deduplicated.jsonl", "--output_path", f"{out_dir}/scraped.jsonl", "--n_total_process", n],
    ["python", "-u", "-m", "utils.validate", "--raw_data_path", f"{out_dir}/scraped.jsonl", "--output_path", f"{out_dir}/validated.jsonl", "--query_data_path", query, "--n_total_process", n],
    ["python", "-u", "-m", "utils.stat", "--input_path", f"{out_dir}/validated.jsonl", "--output_path", f"{out_dir}/fact_result.txt"],
]

with open("fact_grading_log.txt", "w") as log:
    for i, cmd in enumerate(steps):
        log.write(f"\n=== Step {i+1}/5: {cmd[3]} ===\n")
        log.flush()
        ensure_grader_checkout()
        aid = reserve_job("grade")
        try:
            r = subprocess.run(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, env=_grader_env, timeout=3600)
        finally:
            settle_job(aid)
        if r.returncode != 0:
            log.write(f"FAILED at step {i+1} with exit code {r.returncode}\n")
            break

out = ""
rpath = f"{cwd}/{out_dir}/fact_result.txt"
if pathlib.Path(rpath).exists():
    out = open(rpath).read()
else:
    out = f"No results. Download fact_grading_log.txt to debug."
pathlib.Path("fact_status.txt").write_text(out)
out